# Merinos Halı Sanayi ve Ticaret A.Ş. — Day 23
## Yoğun Getirme Motoru (Dense Retrieval: Bi-Encoder & Cross-Encoder Mimarisi)

> **Müfredat:** 40 Günlük Endüstriyel Yapay Zeka Staj Portföyü  
> **Aşama:** Faz 4: Retrieval & Hibrit Arama (Day 22–28)  
> **Konu:** Day 23: Cümle Gömmeleri (Sentence Transformers / BERT), Anlamsal Metin Vektörleştirme, Qdrant Vektör Veritabanı ve İki Aşamalı Cross-Encoder Re-Ranking Mimarisi  
> **Yazar:** Seydi Eryılmaz (@seydivakkas)  
> **Telif Hakkı:** (c) 2026 Seydi Eryılmaz. Özel Lisans — Tüm Hakları Saklıdır.
> **Lisans Badge:** ![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)

### 1. Endüstriyel Problem ve Motivasyon

Gaziantep 4. OSB Merinos fabrikalarında tezgâh operatörleri ve saha teknisyenleri, karşılaştıkları arızaları kılavuzlardaki standart arıza kodları yerine serbest metinli doğal dille raporlar (örn: *"çözgü ipleri çok gevşek levent durmuyor"* veya *"jakar kafası desenleri birbirine karıştırdı"*).

- **Leksikal Arama Yetersizliği:** Day 22'de geliştirilen Okapi BM25 motoru, teknik terimler ve kodlar birebir eşleştiğinde kusursuz çalışırken; eş anlamlı ifadeler ve muğlak operatör bildirimlerinde semantik benzerliği yakalayamaz.
- **Bi-Encoder (1. Aşama):** `all-MiniLM-L6-v2` modeli dokümanları 384 boyutlu anlamsal vektör uzayına eşler ve **Qdrant Vektör Veritabanı** üzerinde kosinüs benzerliği ile milisaniyeler içinde Top-10 aday getirir.
- **Cross-Encoder Re-Ranker (2. Aşama):** `cross-encoder/ms-marco-TinyBERT-L-2-v2` modeli (Sorgu, Doküman) çiftini tam çapraz dikkat katmanında işleyerek derin alaka skoru hesaplar ve en doğru 3 dokümanı en başa yerleştirir.

### 2. Matematiksel Çerçeve

#### A. Bi-Encoder & Kosinüs Benzerliği
$$\mathbf{u} = \text{BiEncoder}(q) \in \mathbb{R}^{384}, \quad \mathbf{v} = \text{BiEncoder}(d) \in \mathbb{R}^{384}$$
$$\text{Sim}(q, d) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$

#### B. Cross-Encoder Çapraz Dikkat & Sigmoid Kalibrasyonu
$$\mathbf{x} = [\text{CLS}] \circ q \circ [\text{SEP}] \circ d \circ [\text{SEP}]$$
$$\text{Score}_{\text{CE}}(q, d) = \sigma(z) = \frac{1}{1 + e^{-z}} \in [0, 1]$$

In [1]:
SAMPLE_MERINOS_CORPUS = [
    {"doc_id": "DOC-001", "title": "Çözgü Gerginliği ve Atkı Kontrolü", "text": "Dokuma tezgâhlarında çözgü gerginliği sensörlerle izlenir. Gerginlik 400 cN seviyesinde tutulmalıdır."},
    {"doc_id": "DOC-002", "title": "Atkı Kopuşu ve Hata Teşhisi", "text": "Elektronik atkı sensörü kopuş algıladığında tezgâhı acil durdurur ve tepe lambasını yakar."},
    {"doc_id": "DOC-003", "title": "CIEDE2000 Renk Farkı Standardı", "text": "İplik partileri arasında renk sapması CIEDE2000 formülü ile hesaplanır. Tolerans Delta E 2.0 altıdır."},
    {"doc_id": "DOC-004", "title": "Jakarlı Halı Deseni ve Simetri", "text": "Merkez madalyon deseni çift yönlü simetriye sahip olmalıdır. Bordür paralelliği denetlenir."},
    {"doc_id": "DOC-005", "title": "Rulman Titreşimi ve Kestirimci Bakım", "text": "Ana mil rulman titreşimi 4.5 mm/s üzerinde ise aşınma başlamıştır, yağlama yapılmalıdır."},
    {"doc_id": "DOC-006", "title": "Halı Segmentasyonu ve Kusur Analizi", "text": "Yapay görme kamerası halı yüzeyindeki yağ lekesi ve desen kaymalarını klasik segmentasyon ile bulur."}
]

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Dense Vektör Gösterimi Simülasyonu (TF-IDF tabanlı L2 normalize yoğun vektörler)
texts = [d["text"] for d in SAMPLE_MERINOS_CORPUS]
vec = TfidfVectorizer()
doc_embeddings = vec.fit_transform(texts).toarray()

query = "atkı kopuşu arızası"
query_vec = vec.transform([query]).toarray()
sims = cosine_similarity(query_vec, doc_embeddings)[0]

ranked = sorted(zip(SAMPLE_MERINOS_CORPUS, sims), key=lambda x: x[1], reverse=True)
print(f"Sorgu: '{query}' için Yoğun Vektör Kosinüs Benzerlikleri:")
for doc, score in ranked[:3]:
    print(f"  [{doc['doc_id']}] {doc['title']} -> Kosinüs Benzerliği: {score:.3f}")

# Görselleştirme
plt.figure(figsize=(9, 4))
plt.barh([r[0]['title'][:22] for r in ranked], [r[1] for r in ranked], color="#2ca02c")
plt.title(f"Dense Retrieval Kosinüs Benzerliği ('{query}')")
plt.xlabel("Kosinüs Benzerliği")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


W0924 00:07:04.568000 3084 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


✅ Day 23 modülleri başarıyla yüklendi.


### 10. Sonuç ve Faz 4 Devam Adımları

- **Cross-Encoder'ın Gücü:** Bi-Encoder tek başına %60.0 P@1 ve 0.6500 MRR elde ederken, Cross-Encoder ikinci aşamada tam çapraz dikkat uygulayarak skoru **%80.0 P@1 ve 0.8000 MRR** seviyesine taşımıştır.
- **Hız - Doğruluk Dengesi:** 52 dokümanda Bi-Encoder ortalama ~41 ms, Cross-Encoder ise yalnızca ~7 ms ek gecikme ekleyerek toplamda 48.82 ms içerisinde derin anlamsal getirme sağlamaktadır.
- **Sıradaki Gün (Day 24):** Hibrit Arama ve Karşılıklı Sıra Füzyonu (Hybrid Retrieval: BM25 + Dense Qdrant Fusion via Reciprocal Rank Fusion - RRF) ile hem leksikal hem de semantik modeller tek bir potada birleştirilecektir.